# 01 baseline MD -- Li6PS5Cl dual-potential comparison, thin pipeline (W5-W6)

Un-fine-tuned baseline: run both **MACE-MP-0** and **MatterSim** as baselines on the same
Li6PS5Cl cell, multi-temperature NVT MD -> sigma(T) -> Arrhenius extrapolation to 300K ->
benchmark against experiment's ~3.15 mS/cm.

**Thin pipeline**: the MD run is short and the statistics are unconverged, so sigma is
only an order-of-magnitude indicator; **baseline** = un-fine-tuned, with an expected 2-40%
bias -- *seeing that bias is exactly the point*; fine-tuning is W7.

**Usage**: Runtime -> Change runtime type -> **T4 GPU** -> create `MP_API_KEY` under the
left-hand 🔑 Secrets panel -> Run all.
Heavy packages (torch/mace/mattersim) run on the Colab GPU; structure building (01) and
analysis (03) can also run locally.

In [ ]:
# 1) Install packages + confirm GPU (mattersim is large, first install takes ~3-5 minutes)
!pip install -q mace-torch mattersim kinisi pymatgen-analysis-diffusion mp-api ase pymatgen scipp
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('CUDA:', torch.cuda.is_available(), '| GPU:', gpu, '| device:', device)
# If this prints False/CPU: Change runtime type -> T4 GPU -> Run all.
# If mattersim fails to install, run MACE only for now (add --mlip mace to 02 below) and add MatterSim later.

In [ ]:
# 2) Get the code: A) set REPO_URL to git clone; B) otherwise upload ai4ssb-mlip-md.zip and auto-extract
import os, glob, zipfile
REPO_URL = ''  # e.g. 'https://github.com/E1582271-dotcom/ai4ssb-mlip-md.git'
if REPO_URL:
    !git clone -q $REPO_URL p2 && echo cloned
    %cd p2
else:
    zf = None
    try:
        from google.colab import files
        print('Please select ~/Desktop/ai4ssb-mlip-md.zip on your machine to upload ...')
        up = files.upload()
        zf = sorted(up)[0]
    except Exception:
        zf = (glob.glob('ai4ssb-mlip-md*.zip') or glob.glob('*.zip') or [None])[0]
    if zf:
        with zipfile.ZipFile(zf) as z:
            z.extractall('.')       # extract into the current directory, scripts are ready to run
        print('Extraction complete:', zf)
    else:
        print('No zip found: drag ai4ssb-mlip-md.zip into the Files panel on the left and re-run this cell, or set REPO_URL.')
print('cwd:', os.getcwd(), '| scripts:', sorted(glob.glob('0*_*.py')))

In [ ]:
# 3) Read MP_API_KEY from a Colab Secret (needed by 01 to pull the real Li6PS5Cl)
import os
try:
    from google.colab import userdata
    os.environ['MP_API_KEY'] = userdata.get('MP_API_KEY')
    print('MP_API_KEY loaded from Colab Secret')
except Exception as e:
    print('Set MP_API_KEY in Colab Secrets (skip 01 if data/ already has the config CIF):', e)

In [ ]:
# 4) Build the structure (CPU): MP mp-985592 -> S/Cl disorder enumeration -> data/config*.cif
!python 01_build_structure.py

In [ ]:
# 5) Baseline multi-temperature MD (thin: 2 temperatures, 30 ps first, to verify it runs, both potentials)
# The first run downloads MACE / MatterSim weights separately. Once it works, scale up to --temps 600,800,1000 --steps 50000.
!python 02_baseline_md.py --temps 600,1000 --steps 30000 --equilib 3000

In [ ]:
# 6) Analyze: MSD -> D (pymatgen + kinisi error bars) -> sigma(T) -> Arrhenius -> benchmark against experiment
!python 03_analyze_transport.py

In [ ]:
# 7) Show figures (baseline outputs are written to figures/supplementary/)
import os
from IPython.display import Image, display
for f in ['figures/supplementary/02_md_stability_mace.png', 'figures/supplementary/02_md_stability_mattersim.png',
          'figures/supplementary/03_arrhenius.png', 'figures/supplementary/03_sigma300_vs_expt.png']:
    if os.path.exists(f):
        print(f); display(Image(f))

## After it runs

1. **Add temperatures + duration**: `--temps 600,800,1000 --steps 50000` (need three points
   to check whether the Arrhenius fit is linear).
2. **Add a supercell**: locally, `01_build_structure.py --supercell 2,2,2` (416 atoms)
   improves statistics, but will be slow on a T4.
3. **W6 verbal checkpoint** (self-check): the bcc sulfur framework / Arrhenius
   extrapolation / why fine-tuning is needed / concerted migration.
4. **W8** production MD (converged 416-atom / 200 ps) ran on the free NUS Vanda A40 queue
   (`hpc/run_md.pbs`); the AutoDL rental budgeted for it was ultimately not needed.

Don't take the thin pipeline's sigma at face value: it verifies "the whole pipeline runs
end to end", not "sigma is accurate". The deviation from experiment is exactly the
material for the baseline diagnosis.